# Importing Libs

In [1]:
!pip install langchain_community
!pip install langchain_huggingface
!pip install langchain_groq
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 71.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.2 MB/s eta 0:00:00


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

# Preprocessing

In [3]:
import json

with open("squad_subset.json") as f:
    data = json.load(f)

contexts = [item["context"] for item in data]
questions = [item["question"] for item in data]
answers = [item["answers"]["text"][0] for item in data]

# Chunking

In [5]:
splitter = RecursiveCharacterTextSplitter( chunk_size=300, chunk_overlap=50)
chunked_contexts = splitter.create_documents(contexts)

# Embeddings

In [4]:
embeddings = HuggingFaceEmbeddings( model_name = "all-MiniLM-L6-V2" )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# VectorStore 

In [6]:
vectorstore = FAISS.from_documents( chunked_contexts, embeddings)

# Retrieval

In [7]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

# LLM

In [17]:
from google.colab import userdata
api_key = userdata.get("groq_api_key_3")


llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=api_key
)

# Rag Func

In [18]:
def simple_rag(question):

    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are a question answering system.

Use ONLY the context to answer.

Rules:
- Return ONLY the short answer span.
- Do NOT explain.
- Do NOT write full sentences.
- Answer must be a phrase from the context.

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)

    return response.content

In [19]:
print(simple_rag(questions[0]))

Denver Broncos


In [20]:
print(simple_rag(questions[197]))

# Evaluation

In [31]:
predictions = []

In [51]:
import time



for q in questions[75:100]:

    while True:
        try:
            pred = simple_rag(q)
            predictions.append(pred)
            time.sleep(6)
            break
        except Exception:
            print("Rate limit hit, waiting...")
            time.sleep(10)

In [52]:
predictions

['Denver Broncos',
 'Carolina Panthers',
 '',
 'Denver Broncos',
 '\n\n',
 '',
 'Tuesday',
 '',
 '',
 '',
 'Monday',
 'Denver Broncos',
 'Justin Herman Plaza',
 '',
 'Super Bowl L',
 'the 2015 season',
 '',
 '',
 "Levi's Stadium",
 '',
 '',
 '2015',
 '',
 'The Panthers',
 'Denver Broncos',
 'the 2015 season',
 'Denver Broncos',
 '',
 'the game',
 'Denver Broncos',
 'Cam Newton',
 'eight',
 '',
 'Seattle Seahawks',
 'Newton',
 'the Arizona Cardinals',
 'New England Patriots',
 '',
 'four',
 'Cam Newton',
 'won at least 15 regular season games',
 '',
 '12–4',
 'four teams',
 '',
 '',
 'the Arizona Cardinals',
 '',
 '',
 'Cam Newton',
 '',
 'Arizona Cardinals',
 'Cam Newton',
 'the Arizona Cardinals',
 '',
 '',
 'two forced fumbles',
 'Denver',
 'Miller',
 'five solo tackles',
 'the Broncos',
 'Newton was sacked',
 'Bart\u202fStarr',
 'one interception',
 'two forced fumbles',
 '',
 '',
 'five',
 'two forced fumbles',
 'Bart\u202fStarr',
 'six total tackles',
 'Newton was sacked',
 '',
 '

In [53]:
len(predictions)

100

In [36]:
print(simple_rag(questions[197]))

## Text Normalisation

In [37]:
import re
import string

def normalize_text(text):
    text = text.lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = ''.join(ch for ch in text if ch not in string.punctuation)
    text = ' '.join(text.split())
    return text

## Exact Match

In [38]:
def exact_match_score(prediction, ground_truth):
    return normalize_text(prediction) == normalize_text(ground_truth)

## F1 Score

In [39]:
def f1_score(prediction, ground_truth):

    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(ground_truth).split()

    common = set(pred_tokens) & set(truth_tokens)

    if len(common) == 0:
        return 0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(truth_tokens)

    return 2 * (precision * recall) / (precision + recall)

## Eval Func

In [40]:
def evaluate(predictions, subset_ans):

    total = len(predictions)
    exact_match = 0
    f1 = 0

    for pred, truth in zip(predictions, subset_ans):

        if exact_match_score(pred, truth):
            exact_match += 1

        f1 += f1_score(pred, truth)

    exact_match = exact_match / total
    f1 = f1 / total

    return {
        "Exact Match": exact_match,
        "F1 Score": f1
    }

In [55]:
subset_ans=answers[:100]
evaluate(predictions, subset_ans)

{'Exact Match': 0.32, 'F1 Score': 0.37133333333333335}